# 02 — Hierarchical Clustering, Dendrograms, and DBSCAN

This notebook compares K-Means, hierarchical agglomerative clustering, and density-based DBSCAN on the Iris feature matrix. Species labels are excluded from fitting and reserved for an optional post-hoc check.

## Learning objectives
- Build and interpret a hierarchical dendrogram.
- Compare K-Means and agglomerative clustering.
- Explore linkage criteria and cluster assignments.
- Run DBSCAN and interpret noise points (`-1`).
- Use silhouette scores carefully when evaluating clustering.

References: [Scikit-learn clustering guide](https://scikit-learn.org/stable/modules/clustering.html) and [SciPy dendrogram API](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.dendrogram.html).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

## 1. Load and scale Iris features
All three algorithms use the same standardized feature matrix so differences are easier to compare.

In [ ]:
iris = load_iris()
X = iris.data
species_labels = iris.target  # Only for post-hoc comparison
X_scaled = StandardScaler().fit_transform(X)
print('Feature matrix:', X_scaled.shape)

## 2. Hierarchical clustering with a dendrogram
The dendrogram shows which observations or groups merge and the distance at which they merge. A horizontal cut through the tree yields a flat partition. SciPy is used for the dendrogram; Scikit-learn provides the clustering estimator.

In [ ]:
linkage_matrix = linkage(X_scaled, method='ward', metric='euclidean')

plt.figure(figsize=(14, 6))
dendrogram(linkage_matrix, no_labels=True, color_threshold=None)
plt.xlabel('Iris observations')
plt.ylabel('Ward merge distance')
plt.title('Hierarchical Clustering Dendrogram (Ward linkage)')
plt.tight_layout()
plt.show()

### Create a flat partition from the hierarchy
For demonstration, cut the hierarchy to obtain up to three clusters. This is a chosen teaching setting, not a universal rule.

In [ ]:
hierarchical_labels = fcluster(linkage_matrix, t=3, criterion='maxclust') - 1
print(pd.Series(hierarchical_labels).value_counts().sort_index())
print('Silhouette score:', round(silhouette_score(X_scaled, hierarchical_labels), 3))

## 3. AgglomerativeClustering API
Scikit-learn's `AgglomerativeClustering` returns flat cluster labels. Ward linkage minimizes the increase in within-cluster variance and uses Euclidean distance.

In [ ]:
agglomerative = AgglomerativeClustering(n_clusters=3, linkage='ward')
agglomerative_labels = agglomerative.fit_predict(X_scaled)
print('Agglomerative silhouette:', round(silhouette_score(X_scaled, agglomerative_labels), 3))
print(pd.Series(agglomerative_labels).value_counts().sort_index())

## 4. Compare K-Means and hierarchical clustering
Both methods can return a flat partition, but they construct clusters differently. Numeric cluster IDs from different algorithms are arbitrary and should not be treated as equivalent.

In [ ]:
kmeans_labels = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X_scaled)

method_comparison = pd.DataFrame({
    'Method': ['K-Means', 'Agglomerative Ward', 'SciPy Ward cut'],
    'Silhouette score': [
        silhouette_score(X_scaled, kmeans_labels),
        silhouette_score(X_scaled, agglomerative_labels),
        silhouette_score(X_scaled, hierarchical_labels),
    ],
    'Number of clusters': [len(np.unique(labels)) for labels in [kmeans_labels, agglomerative_labels, hierarchical_labels]],
})
display(method_comparison.round(3))

### Comparison notes
- **K-Means** repeatedly updates centroids and assignments; it requires `n_clusters` in advance.
- **Agglomerative clustering** repeatedly merges clusters according to a linkage rule.
- **Dendrograms** show the merge hierarchy and support exploration of different cut heights.
- Silhouette scores should be interpreted alongside the cluster geometry and plots.

## 5. DBSCAN clustering
DBSCAN grows clusters from dense neighborhoods. It does not require a cluster count in advance and labels noise observations as `-1`. `eps` is a radius in standardized feature space; `min_samples` controls the density requirement.

Official API: [sklearn.cluster.DBSCAN](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html).

In [ ]:
dbscan = DBSCAN(eps=0.7, min_samples=5, metric='euclidean')
dbscan_labels = dbscan.fit_predict(X_scaled)

unique_labels, counts = np.unique(dbscan_labels, return_counts=True)
print('DBSCAN label counts (label -1 means noise):')
print(dict(zip(unique_labels.tolist(), counts.tolist())))

n_clusters = len(set(dbscan_labels) - {-1})
n_noise = int(np.sum(dbscan_labels == -1))
print(f'Clusters excluding noise: {n_clusters}')
print(f'Noise samples: {n_noise}')

### Visualize DBSCAN clusters
PCA displays the standardized four-feature dataset in two dimensions. DBSCAN was fit on all four standardized features.

In [ ]:
X_pca = PCA(n_components=2, random_state=42).fit_transform(X_scaled)
plt.figure(figsize=(8, 5))
for label in sorted(np.unique(dbscan_labels)):
    mask = dbscan_labels == label
    label_name = 'Noise (-1)' if label == -1 else f'Cluster {label}'
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], s=42, alpha=0.8, label=label_name)
plt.xlabel('PCA component 1')
plt.ylabel('PCA component 2')
plt.title('DBSCAN Clusters and Noise (PCA projection)')
plt.legend()
plt.grid(alpha=0.15)
plt.show()

### Silhouette score for DBSCAN (when valid)
Silhouette requires at least two clusters and fewer clusters than samples. Noise points are excluded from this calculation. If DBSCAN returns fewer than two non-noise clusters, the score is undefined.

In [ ]:
non_noise_mask = dbscan_labels != -1
non_noise_labels = dbscan_labels[non_noise_mask]
n_non_noise_clusters = len(np.unique(non_noise_labels))

if 2 <= n_non_noise_clusters < int(non_noise_mask.sum()):
    score = silhouette_score(X_scaled[non_noise_mask], non_noise_labels)
    print('Silhouette score on non-noise samples:', round(score, 3))
else:
    print('Silhouette score is undefined for this DBSCAN result: need at least 2 non-noise clusters and fewer clusters than samples.')

## Key takeaways
- Hierarchical clustering provides a hierarchy; a dendrogram helps explore possible cuts.
- K-Means and agglomerative clustering have different assumptions and can produce different partitions.
- DBSCAN finds dense regions and identifies noise, but results depend on `eps`, `min_samples`, and feature scaling.
- Inspect cluster plots and data characteristics alongside internal scores.
- Cluster IDs have no inherent species meaning.